# Hands-on 7: Membersihkan Timestamp, Resampling, dan Imputasi Sederhana

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 4.

Alur: parse timestamp -> buang invalid -> urutkan -> hapus duplikat -> resample 10 menit ->
imputasi (forward fill vs interpolasi linier) -> simpan `data/processed/room_sensor_clean.csv`.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Bangkitkan Dataset Kotor (masalah khas IoT)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/generate_dirty_dataset.py"],
                   capture_output=True, text=True)
print(r.stdout)

[SUKSES] Dataset kotor tersimpan: data/raw/room_sensor_dirty.csv (1003 baris)
[INFO] Masalah disuntikkan - invalid_timestamp: 20
[INFO] Masalah disuntikkan - duplikat: 15
[INFO] Masalah disuntikkan - nilai_hilang: 120
[INFO] Masalah disuntikkan - baris_gap: 20



## 2. Cleaning, Resampling 10 Menit, dan Imputasi

In [3]:
r = subprocess.run([sys.executable, "scripts/clean_timestamp_resample.py"],
                   capture_output=True, text=True)
print(r.stdout)

[INFO] Baris awal: 1003
[INFO] Timestamp invalid dibuang: 20
[INFO] Duplikasi timestamp dibuang: 15
[INFO] Baris dengan nilai hilang setelah resample: 148

=== Data setelah resampling 10 menit ===
                     temperature_c  humidity_pct  co2_ppm  power_watt device_id
timestamp                                                                      
2026-01-01 00:00:00          23.99         66.76    431.0       85.57    room01
2026-01-01 00:10:00          23.98         69.47    457.0       83.39    room01
2026-01-01 00:20:00          23.64         67.00    431.0       90.09    room01
2026-01-01 00:30:00          23.24         68.95    437.0       90.72    room01
2026-01-01 00:40:00          23.32         69.79    453.0       82.10    room01
2026-01-01 00:50:00            NaN           NaN      NaN         NaN    room01
2026-01-01 01:00:00          23.77         67.30    443.0       83.55    room01
2026-01-01 01:10:00          23.62         70.32    454.0       93.68    room01
202

## 3. Perbandingan Visual Sebelum/Sesudah

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

df_dirty = pd.read_csv("data/raw/room_sensor_dirty.csv")
df_dirty["timestamp"] = pd.to_datetime(df_dirty["timestamp"], errors="coerce")
df_dirty = df_dirty.dropna(subset=["timestamp"]).sort_values("timestamp")
df_clean = pd.read_csv("data/processed/room_sensor_clean.csv", parse_dates=["timestamp"])

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(df_dirty["timestamp"], df_dirty["temperature_c"], ".", ms=2, color="tomato")
axes[0].set_title("Sebelum Cleaning - Suhu (timestamp kotor, duplikat, gap, NaN)")
axes[0].set_ylabel("Suhu (C)")
axes[1].plot(df_clean["timestamp"], df_clean["temperature_c"], "-", lw=1, color="seagreen")
axes[1].set_title("Sesudah Cleaning - Resample 10 menit + Interpolasi Linier")
axes[1].set_ylabel("Suhu (C)")
axes[1].set_xlabel("Waktu")
plt.xticks(rotation=45)
plt.tight_layout()
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("outputs/figures/cleaning_before_after.png")
plt.show()
print("[SUKSES] outputs/figures/cleaning_before_after.png")

[SUKSES] outputs/figures/cleaning_before_after.png


## Penutup

Data kotor (timestamp invalid, duplikat, nilai hilang, gap interval) berhasil dibersihkan
menjadi deret 10 menit yang konsisten tanpa nilai hilang. Dataset bersih ini menjadi dasar
deteksi anomali statistik pada Hands-on 8.

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 7 dalam urutan buku) agar dapat dijadikan panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 10. Langkah kerja yang harus dilakukan pembaca

Tahap

Kegiatan

Baca file kotor.

Parse timestamp.

Buang waktu invalid.

Urutkan data dan hapus duplikasi.

Set index waktu.

Lakukan resample ke interval target.

Lakukan imputasi sederhana.

Simpan hasil ke folder processed.